In [48]:
import os
import shutil
from google.colab import drive

# 1. Safely unmount and flush any pending writes
try:
    drive.flush_and_unmount()
    print("Successfully unmounted active Drive.")
except Exception as e:
    print(f"Drive was likely not mounted: {e}")

# 2. Programmatically clear the Google DriveFS authentication cache
cache_path = '/root/.config/Google/DriveFS'
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Authentication cache cleared successfully.")
else:
    print("No authentication cache found. Ready for a clean mount.")

# 3. Trigger a fresh authentication request
print("\nOpening authentication prompt for your new account...")
drive.mount('/content/drive', force_remount=True)

Successfully unmounted active Drive.
Authentication cache cleared successfully.

Opening authentication prompt for your new account...
Mounted at /content/drive


In [12]:
from google.colab import files

files.upload()

Saving kaggle.json to kaggle.json


OSError: [Errno 107] Transport endpoint is not connected: 'kaggle.json'

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d awsaf49/brats20-dataset-training-validation
!unzip -q brats20-dataset-training-validation.zip -d /content

Dataset URL: https://www.kaggle.com/datasets/awsaf49/brats20-dataset-training-validation
License(s): CC0-1.0
100% 4.16G/4.16G [00:42<00:00, 105MB/s]



In [1]:
!pip install nibabel albumentations transformers scikit-learn

In [2]:
%cd /content/drive/MyDrive/PUBLICATION_READY_CODE

!find . -maxdepth 2 -type f

/content/drive/MyDrive/PUBLICATION_READY_CODE
./requirements.txt
./__pycache__/model.cpython-312.pyc
./configs/config.py
./configs/__init__.py
./data/__init__.py
./data/transforms.py
./data/brats_dataset.py
./data/build.py
./models/__init__.py
./models/dino_encoder.py
./models/prototype.py
./models/transformer_refiner.py
./models/decoder.py
./models/model.py
./metrics/segmentation_metrics.py
./metrics/__init__.py
./utils/checkpoint.py
./utils/history.py
./utils/seed.py
./losses/losses.py
./losses/__init__.py


In [3]:
import os
import sys
from data.build import build_dataloaders
from configs.config import *


print(os.listdir("/content/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"))


sys.path.append("/content/drive/MyDrive/PUBLICATION_READY_CODE")

train_loader, val_loader, test_loader = build_dataloaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    image_size=IMAGE_SIZE
)

print("\n===== Dataset Statistics =====")
print("Train samples :", len(train_loader.dataset))
print("Val samples   :", len(val_loader.dataset))
print("Test samples  :", len(test_loader.dataset))

images, masks = next(iter(train_loader))

print("\n===== Batch Shapes =====")
print("Images:", images.shape)
print("Masks :", masks.shape)

print("\n===== Mask Values =====")
print("Unique values:", masks.unique())


['BraTS20_Training_222', 'BraTS20_Training_050', 'BraTS20_Training_047', 'BraTS20_Training_182', 'BraTS20_Training_052', 'BraTS20_Training_145', 'BraTS20_Training_008', 'BraTS20_Training_003', 'BraTS20_Training_367', 'BraTS20_Training_057', 'BraTS20_Training_025', 'BraTS20_Training_247', 'BraTS20_Training_269', 'BraTS20_Training_327', 'BraTS20_Training_285', 'BraTS20_Training_303', 'BraTS20_Training_005', 'BraTS20_Training_122', 'BraTS20_Training_096', 'BraTS20_Training_201', 'BraTS20_Training_062', 'BraTS20_Training_184', 'BraTS20_Training_181', 'BraTS20_Training_238', 'BraTS20_Training_033', 'BraTS20_Training_359', 'BraTS20_Training_243', 'BraTS20_Training_318', 'BraTS20_Training_027', 'BraTS20_Training_312', 'BraTS20_Training_301', 'BraTS20_Training_225', 'BraTS20_Training_193', 'BraTS20_Training_340', 'BraTS20_Training_278', 'BraTS20_Training_068', 'BraTS20_Training_292', 'BraTS20_Training_236', 'BraTS20_Training_341', 'BraTS20_Training_154', 'BraTS20_Training_277', 'BraTS20_Traini

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



===== Batch Shapes =====
Images: torch.Size([2, 1, 224, 224])
Masks : torch.Size([2, 1, 224, 224])

===== Mask Values =====
Unique values: tensor([0., 1.])


In [4]:
import torch

from models.model import DINOProtoFormer

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = DINOProtoFormer().to(device)

x = torch.randn(
    2,
    1,
    224,
    224
).to(device)

with torch.no_grad():
    y = model(x)

print("Input :", x.shape)
print("Output:", y.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Input : torch.Size([2, 1, 224, 224])
Output: torch.Size([2, 1, 224, 224])


In [5]:
import os

project = "/content/drive/MyDrive/PUBLICATION_READY_CODE"

# losses
if os.path.exists(f"{project}/losses/init.py"):
    os.rename(
        f"{project}/losses/init.py",
        f"{project}/losses/__init__.py"
    )

# metrics
if os.path.exists(f"{project}/metrics/init.py"):
    os.rename(
        f"{project}/metrics/init.py",
        f"{project}/metrics/__init__.py"
    )

print("Fixed __init__.py files")

Fixed __init__.py files


In [6]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/PUBLICATION_READY_CODE"

# Remove if already present
if PROJECT_ROOT in sys.path:
    sys.path.remove(PROJECT_ROOT)

# Insert at the beginning
sys.path.insert(0, PROJECT_ROOT)

print("sys.path[0] =", sys.path[0])

!ls losses
!ls metrics

!mv losses/init.py losses/__init__.py
!mv metrics/init.py metrics/__init__.py

sys.path[0] = /content/drive/MyDrive/PUBLICATION_READY_CODE
__init__.py  losses.py
__init__.py  segmentation_metrics.py
mv: cannot stat 'losses/init.py': No such file or directory
mv: cannot stat 'metrics/init.py': No such file or directory


In [7]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/PUBLICATION_READY_CODE

import sys

PROJECT_ROOT = "/content/drive/MyDrive/PUBLICATION_READY_CODE"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/PUBLICATION_READY_CODE


In [8]:
import torch

from losses.losses import DiceBCELoss
from metrics.segmentation_metrics import dice_score

criterion = DiceBCELoss()

pred = torch.randn(2, 1, 224, 224)

target = (torch.rand(2, 1, 224, 224) > 0.5).float()

loss = criterion(pred, target)
dice = dice_score(pred, target)

print("Loss :", loss.item())
print("Dice :", dice)

Loss : 0.653611421585083
Dice : 0.4990766942501068


In [9]:
import os
import sys

print("Current directory:", os.getcwd())
print("\nsys.path[:5]:")
print(sys.path[:5])

print("\nLosses folder:")
print(os.listdir("/content/drive/MyDrive/PUBLICATION_READY_CODE/losses"))

print("\nMetrics folder:")
print(os.listdir("/content/drive/MyDrive/PUBLICATION_READY_CODE/metrics"))

Current directory: /content/drive/MyDrive/PUBLICATION_READY_CODE

sys.path[:5]:
['/content/drive/MyDrive/PUBLICATION_READY_CODE', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12']

Losses folder:
['losses.py', '__init__.py', '__pycache__']

Metrics folder:
['segmentation_metrics.py', '__init__.py', '__pycache__']


In [ ]:
%cd /content/drive/MyDrive/PUBLICATION_READY_CODE
!python -u train.py

/content/drive/MyDrive/PUBLICATION_READY_CODE
Building DataLoaders...
Tumor slices: 19452
Tumor slices: 2416
Tumor slices: 2486

Dataset Statistics
------------------------------------------------------------
Train samples : 19452
Val samples   : 2416
Test samples  : 2486

Train batches : 9726
Val batches   : 1208
Test batches  : 1243

Building Model...
Loading weights: 100% 223/223 [00:00<00:00, 13289.33it/s]
DINOProtoFormer

Total Parameters     : 26,785,025
Trainable Parameters : 4,728,449

Device: cpu
/content/drive/MyDrive/PUBLICATION_READY_CODE/train.py:173: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(

AMP Enabled: False

Initialization Complete.

Epoch 1/50
Training:   0% 0/9726 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory w

In [ ]:
import os
import sys
import json
from tqdm import tqdm

import torch
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler

# =====================================================
# PROJECT ROOT
# =====================================================

PROJECT_ROOT = "/content/drive/MyDrive/PUBLICATION_READY_CODE"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    # =====================================================
    # SAVE TRAINING HISTORY
    # =====================================================

    try:

        with open(HISTORY_PATH, "w") as f:

            json.dump(
                history,
                f,
                indent=4
            )

    except Exception as e:

        print(
            "Could not save history:",
            e
        )

    # =====================================================
    # SAVE LATEST CHECKPOINT
    # =====================================================

    try:

        save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            best_dice=best_dice,
            path=LATEST_MODEL_PATH
        )

        print(
            "Latest checkpoint saved."
        )

    except Exception as e:

        print(
            "Could not save latest checkpoint:",
            e
        )

    # =====================================================
    # SAVE BEST MODEL
    # =====================================================

    if epoch_val_dice > best_dice:

        best_dice = epoch_val_dice

        patience_counter = 0

        try:

            save_checkpoint(
                model=model,
                optimizer=optimizer,
                epoch=epoch,
                best_dice=best_dice,
                path=BEST_MODEL_PATH
            )

            print(
                f"New Best Dice: "
                f"{best_dice:.4f}"
            )

            print(
                "Best model saved."
            )

        except Exception as e:

            print(
                "Could not save best checkpoint:",
                e
            )

    else:

        patience_counter += 1

        print(
            f"No improvement "
            f"({patience_counter}/"
            f"{EARLY_STOPPING})"
        )

    # =====================================================
    # EARLY STOPPING
    # =====================================================

    if patience_counter >= EARLY_STOPPING:

        print("\n" + "=" * 60)

        print(
            "Early stopping triggered."
        )

        print(
            f"Best Validation Dice: "
            f"{best_dice:.4f}"
        )

        print("=" * 60)

        break


# =====================================================
# TRAINING COMPLETE
# =====================================================

print("\n" + "=" * 60)

print("Training Completed")

print(
    f"Best Dice = "
    f"{best_dice:.4f}"
)

print(
    f"History Saved To:\n"
    f"{HISTORY_PATH}"
)

print(
    f"\nBest Model:\n"
    f"{BEST_MODEL_PATH}"
)

print(
    f"\nLatest Model:\n"
    f"{LATEST_MODEL_PATH}"
)

print("=" * 60)

# =====================================================
# FINAL GPU STATUS
# =====================================================

if torch.cuda.is_available():

    print(
        "Final GPU Memory:",
        round(
            torch.cuda.memory_allocated()
            / 1024**3,
            2
        ),
        "GB"
    )

    torch.cuda.empty_cache()